Enabling AI-Powered Business Intelligence for Organizations.<br>

Problem scenario:<br>
In today’s data-centric business environment, organizations across various industries accumulate vast amounts of information. However, many struggle to transform this data into actionable insights, especially small to medium-sized enterprises that lack the resources for advanced business intelligence tools.<br>

Recent advancements in artificial intelligence, especially in Large Language Models (LLMs) and Retrieval-Augmented Generation (RAG) systems, offer immense potential for data analysis and insight generation.

Project objective:<br>
InsightForge, an innovative Business Intelligence Assistant, aims to address these challenges by developing an automated AI model using advanced technologies, including LangChain, Retrieval-Augmented Generation (RAG), and Large Language Models (LLMs).<br>

This model aims to:<br>
Analyze User Input and understand the Intent of User Query and then mak decision to Search with RAG on PDF Documents or run a Query on Pandas dataset to get the information from Sales Data. 

Solution Steps: <br>
1. Sales Data is Loaded into Pandas.
2. Vector Search is enabled woth help of FAISS 
3. The Solution starting point is the understanding about User Intent.
4. Based on User Query - if the Intent is 'Research' then Vector DB is lookedup and result is used in RAG.
5. Based on User Query - If the Intent is 'DBLookup' then Pandas Query is generated by the LLM to lookup Pandas.  
6. LLM Model used for User Intent & Pandas Query - "Qwen2.5-Coder-7B-Instruct_q4_K_M".
7. LLM Model used for RAG - llama3.1:8b-instruct-q4_K_M
8. For User Interface we have used traditional Python User Input.
9. All User and Assistant Response are published to Console.  
10. AI liberaries used - LangChain, Ollama, LlamaCloud (PDF Parser), FAISS, RAG


Test Samples. <br>
1. User Input:  
    - Find the Total Count of Males who purchased each category of Products.
   AI Answer:
    - Generated Query <br>
        male_purchases = sales_df[(sales_df['Customer_Gender'] == 'Male')] <br>
        product_counts = male_purchases.groupby('Product').size() <br>
        result = product_counts.to_dict()
    - DBLookup:  {'Widget A': 330, 'Widget B': 303, 'Widget C': 304, 'Widget D': 307}

2. User Input:  
    - What type of data was collectd to build the Time Series Predictions ?
    - Research:  The type of data collected to build the Time Series Predictions includes sensor data from gas sensors (MQ135) and microcontroller (Arduino-Uno). Specifically, the paper mentions collecting data on carbon monoxide (CO), carbon dioxide (CO2), ammonia (NH3), and acetone ((CH3)2CO) levels at three specific locations in Delhi and the National Capital Region (NCR).

3. User Input: 
    - Which Region in Sales Data had highest Year-on-Year growth in 2023 ?
    - Generated Query:  
        sales_2022 = sales_df[sales_df['Date'].str.contains('/2022')] <br>
        sales_2023 = sales_df[sales_df['Date'].str.contains('/2023')] <br>
        growth_by_region = (sales_2023.groupby('Region')['Sales'].sum() - 
                    sales_2022.groupby('Region')['Sales'].sum()) / sales_2022.groupby('Region')['Sales'].sum() <br>
        highest_growth_region = growth_by_region.idxmax() <br>
        highest_growth_value = growth_by_region.max().item() <br>
        result = {'Region': highest_growth_region, 'Growth': float(highest_growth_value)}
    - DBLookup:  {'Region': 'North', 'Growth': 0.12676085644728355}

In [1]:
# Import System Packages
import os
import keyboard
from dotenv import load_dotenv
from pathlib import Path

# Import Pandas
import pandas as pd

# Import Langchain Agents Modules.
from langchain.tools import tool
from langchain.agents import create_agent
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage


# Import Langchain Ollama Chatbot Interfaces
from langchain_ollama import ChatOllama

# Import Sentence Embedding Transformer
import textwrap as tw
from sentence_transformers import SentenceTransformer, CrossEncoder

# Import RAG DB FAISS
import faiss
# from llama_parse import LlamaParse
from llama_cloud import LlamaCloud
from llama_index.core import Document, StorageContext, VectorStoreIndex
from llama_index.core.node_parser import SentenceSplitter   # MarkdownNodeParser is efficient if expand[..] is markdown  
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.faiss import FaissVectorStore


# Huggingface Cache Folder
hf_cache_folder = os.path.expanduser("~/.cache/huggingface/hub")

# Load Env Info
load_dotenv()


c:\Products\Anaconda3\envs\hf1_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

User Intent-1:
- Research PDF Documents - Query Research Database Hosted as Vector Store and Generate RAG Response.

In [2]:
# Parse Documents (PDF, Doc etc...)
# Define Document Parser and Chunkng Strategy

llamaClient = LlamaCloud(
    api_key = os.getenv("LLAMA_CLOUD_API_KEY")
)

# Define Chunking Strategy
chunk_strategy = SentenceSplitter(
    chunk_size=500,
    chunk_overlap=50
)


In [3]:
# Create Embedding for the Document Nodes
# Instantiate FAISS Vector Store
# Instantiate StorageContext associated with LlamaParse
# Store Doc node in FAISS using VectorStoreIndex of LlamaParse

embedding_llm = HuggingFaceEmbedding(
    model_name = "BAAI/bge-large-en-v1.5",
    cache_folder = hf_cache_folder
)

# Instantiate FAISS with Embedding Dimensions
dimension = 1024       #  Dimension BAAI Large(1024) and BAAI Small(384)
faiss_index = faiss.IndexFlatL2(dimension)
faiss_vector_store = FaissVectorStore(faiss_index = faiss_index)

# Instantiate LlamaParse Storage Context
storage_context = StorageContext.from_defaults(
    vector_store = faiss_vector_store
)

llama_vector_store_index = VectorStoreIndex(
    nodes = [],
    embed_model = embedding_llm,
    storage_context = storage_context
)

# Create a Retriever Engine for Simantic Search to get the Doc Chunk, Scores & Metadata
# as_query_engine function does not provide the Simlarity Scores, but provide final Answer.
# query_engine = llama_vector_store_index.as_query_engine(similarity_top_k=5)
query_engine = llama_vector_store_index.as_retriever(similarity_top_k=10)


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 7196.81it/s]


In [4]:
# Create RAG Bot for Generate LLM Response.

# Initialized the RAG LLM (Qwen 7B Q_4_K_M)
def create_rag_llm(model_name: str) -> ChatOllama:
    
    ol_rag_llm = ChatOllama(
        model = model_name,
        temperature = 0.0,
        num_predict = 8192,
        num_ctx = 8192,
        model_kwargs = {
            "repeat_penalty": 1.1,
            "options": {
                "use_cache": False,
                "use_mmap": False
            }
        }
    )

    return(ol_rag_llm)

In [5]:
# Insert to Vector Store Funcation
# Retrieve from Vector Store Funcation

# Insert Individual Doc Sections / Pages that was Parsed.
def insert_parsed_doc(file_name: str, page_num: int, doc_text: str):
    try:
        
        # Wrap Doc / Pages into LlamaIndex Doc.
        llama_doc = Document(
            text = doc_text,
            metadata = {
                "file_name": file_name,
                "page_number": page_num
            }
        )

        # Instert Docuemtns to FAISS
        doc_chunks = chunk_strategy.get_nodes_from_documents([llama_doc])   # GetNode Function need list as argument.
        llama_vector_store_index.insert_nodes(doc_chunks)
        
        total_embedding = llama_vector_store_index.storage_context.vector_store._faiss_index.ntotal
        print("Total Embddings: ", total_embedding, flush=True)

        return(True)
    
    except Exception as exp:
        print("Error Insert Doc: ", type(exp).__name__)
        return(False)


# Retrieve Relevent Chunks from the Vector Store.
def retrieve_doc_chunks(query):
    try:
        resp_chunks = query_engine.retrieve(query)

        #for chunk in resp_chunks:
        #    print("Retrieve Score: ", chunk.score)
        #    print("Retrieve Metadata: ", chunk.metadata.get("file_name"), ":", chunk.metadata.get("page_number"))
        #    print("Retrieve Text: ", chunk.text)
        #    print(resp_chunks[0].text)
        
        return(resp_chunks)
    
    except Exception as exp:
        return(f"Error Retrieve Doc: {exp}")


# Instantiate HF Cross-Encoder Model     
# Setup Re-Ranking after Chunk Retrieval. 
# Respond with a List Chunks

cross_encoder = CrossEncoder(
    model_name_or_path = "cross-encoder/ms-marco-MiniLM-L-6-v2",
    cache_folder = hf_cache_folder
)

def rerank_doc_chunks(query: str, doc_chunks: list) -> list:
    try:

        # Create a Query & Chunk Pair for every Chunk
        doc_pair = []
        for chunk in doc_chunks:
            doc_pair.append([query, chunk.text])

        rerank_scores = cross_encoder.predict(
            inputs = doc_pair,
            device = "cpu",
            convert_to_numpy = True
        )

        # print("Rerank Score: ", rerank_scores, flush=True)
        
        return(rerank_scores)
    
    except Exception as exp:
        return(f"Error Re-Ranking: {exp}")


The Transformer `cache_dir` argument is deprecated. Please pass `cache_dir` via `model_kwargs`, `processor_kwargs`, and/or `config_kwargs` instead.
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 11142.65it/s]


In [6]:
# Compose & Generate Response from selected Chunks 
# Initialise Qwen 7B Quantized LLM using Ollama
# Use the Local LLM to Generate the Response

def format_doc_chunks(query: str, chunks: list) -> str:
    try:
        ctx_list = []
        ctx_template = "Document Chunk-{}: {}"
        
        for i, chunk in enumerate(chunks):
            ctx_list.append(ctx_template.format(i+1, chunk))

        # Formatting Query & Contexts
        context_str = "Retrieved Contexts: \n"
        context_str = context_str + "\n".join(ctx_list) + "\n\n"
        context_str = context_str + "Question: " + query

        # print("RAG Context: \n", context_str)

        return(context_str)
    
    except Exception as exp:
        return(f"Error formatting RAG Context: {exp}")


# Generate RAG Response.
def generate_rag_response(query: str, doc_chunks: list, ragbot: ChatOllama) -> str:

    chat_chain = []

    system_prompt = """
    1. You are a Research Assistant.
    2. Use ONLY the information provided in the Retrieved Context to provide a Response.
    3. If the answer cannot be found in the Retrieved Contexts, say: 'I could not find that information'
    
    Critical Rules for RAG Response:
    1. Do not make up facts.
    2. Do not use Outside Knowledge.
    3. If multiple retrieved documents contain relevant information, combine them into a single coherent answer.
    4. Provide response to Question in 'Answer' variable.
    """

    rag_context = format_doc_chunks(query, doc_chunks)

    chat_chain = [SystemMessage(content=system_prompt)]
    chat_chain.append(HumanMessage(content=rag_context))

    ai_message = (ragbot.invoke(chat_chain).content).split("<|im_start|>assistant")[-1]

    # print("RAG AI Message: ", ai_message)
    
    return(ai_message)


User Intent-2:
- Query SalesData Hosted on Pandas.

In [7]:
# Initialise Query Bot using Ollama

# Create Sales Data HF Chatbot
def create_salesquery_llm(model_name: str) -> ChatOllama:
    
    ol_dfquery_llm = ChatOllama(
        model = model_name,
        temperature = 0.0,
        num_predict = 2048,
        num_ctx = 2048,
        model_kwargs = {
            "repeat_penalty": 1.1,
            "options": {
                "use_cache": False,
                "use_mmap": False
            }
        }
    )

    return(ol_dfquery_llm)


In [8]:
# Format DF Query Messages - System, Human & AI

def format_query_system_messages(chat_hist, sys_msg):
    chat_hist = [
        SystemMessage(content=sys_msg)
    ]
    return(chat_hist)

def format_query_user_messages(chat_hist, usr_msg):
    chat_hist.append(HumanMessage(content=usr_msg))
    return(chat_hist)

def format_query_assistant_messages(ai_msg):
    ai_msg = ai_msg.split("<|im_start|>assistant")[-1]
    clean_msg = ai_msg.replace("```python", "")
    clean_msg = clean_msg.replace("```", "")
    clean_msg = tw.dedent(clean_msg)
    return(clean_msg.strip())

In [9]:
# Extract Info from Sales Data based on User Query over NLP.
# Generate the Pandas Query from User Query(NL).

def query_sales_data(user_query: str, sales_data: pd, df_querybot: ChatOllama) -> str:
    
    chat_chain = []

    system_prompt = """
    General Instructions:
    1. You are a Strict Pandas Query Generator.
    2. Sales Data has been loaded to Pandas DataFrame variable pre-initialized as `sales_df`.
    3. Sales Data has following Schema: Date(MM/DD/YYYY), Product (String), Region (String), Sales (Number), Customer_Age (Number), Customer_Gender (String), Customer_Satisfaction (Number).

    Critical Query Generation Rules:
    1. EXECUTABLE SYNTAX ONLY: Your response must consist solely of executable Python code.
    2. NO MARKDOWN BLOCK WRAPPERS: DO NOT include conversational text, notes, or explanations.
    3. MULTI-LINE ALLOWED: You may write multiple lines of pandas logic if necessary.
    4. NO EXTERNAL IMPORTS: Do not add lines for import pandas as pd or import numpy as np. Assume they are already initialized in the execution sandbox.
    5. ALWAYS ASSIGN the FINAL TRANSFORMED Data or Calculated Data to a VARIABLE Named 'result'.
    6. ALWAYS CAST the Values in 'result' to NATIVE Python type using .item() or int() / float() / str() as appropriate.

    Example Output format for Query with Single Variable Request:
      sales_filter = sales_df[(sales_df['Date'] >= '1/1/2025') & (sales_df['Date'] <= '4/1/2025')]
      total_sales = sales_filter['Sales'].sum()
      result = {'Total_Sales': float(total_sales.item())}

    Example Output format for Query with Multiple Variable Request:
      sales_filter = sales_df[sales_df['Customer_Gender' == 'Female']]
      max_age = sales_filter['Customer_Age'].max()
      min_age = sales_filter['Customer_Age'].min()
      result = {'Max_Age': int(max_age.item()), 'Min_Age': int(min_age.item())}
    """

    chat_chain = format_query_system_messages(chat_chain, system_prompt)
    chat_chain = format_query_user_messages(chat_chain, user_query)
    
    ai_response = format_query_assistant_messages(df_querybot.invoke(chat_chain).content)
    print("DataFrame Query: ", ai_response)

    # Execute Query over DataFrame
    sandbox = {"sales_df": sales_data}
    exec(ai_response, {}, sandbox)
    response = sandbox.get("result")

    return(response)


Build an Agent that Orchestrates Across Intents.
- If the Query is for looking-up Info from a Set of Documents, Search the Vector Store and Retrive the relevent information.
- If the Query is lookup Information from Sales Database, Search the Pandas datastore.

In [10]:
# Create Ollama Agent Chatbot

def create_biagent_llm(model_name: str) -> ChatOllama:

    ol_biagent_llm = ChatOllama(
        model = model_name,
        temperature = 0.0,
        num_predict = 512,
        num_ctx = 2048,
        model_kwargs = {
            "repeat_penalty": 1.1,
            "options": {
                "use_cache": False,
                "use_mmap": False
            }
        }   
    )

    return(ol_biagent_llm)

In [ ]:
# Create Agent Chatbot
# Create Tools for Agent Chatbot

# Research Papers Search Tool
def search_research_data(user_query: str, ragbot: ChatOllama) -> str:
    """ Search the Research Database to Answer the User Query related to various Research topic.
    Args:
        user_query: The exact question or request from user in plain text.
    Return: 
        response: A short factual answer for the question from the vector store.
    """
    print("Inside Search")

    # Retrieve Relevent Chunks from Vector Store
    retrieved_chunks = retrieve_doc_chunks(user_query)

    # Format Chunks with Re-Rank Relevance Score & Sort Descend
    rerank_scores = rerank_doc_chunks(user_query, retrieved_chunks)
    chunk_relevences = [(retrieved_chunk, float(rerank_score)) 
                        for retrieved_chunk, rerank_score in zip(retrieved_chunks, rerank_scores)
    ]
    chunk_relevences.sort(key=lambda item: item[1], reverse=True)

    # print("Relevent Chunks: ", list(chunk_relevences))

    # Choose Chunks with Top 3 Scores
    j = 0
    top3_chunks = []
    for chunk, score in chunk_relevences:
        if ((j < 5)):
            top3_chunks.append(chunk.text)
            j = j + 1

    # Invoke Generate RAG Response
    rag_response = generate_rag_response(user_query, top3_chunks, ragbot)

    return(rag_response)
    

# Sales Data Lookup Tool
def lookup_sales_data(user_query: str, sales_data: pd, df_querybot: ChatOllama) -> str:
    """ Query the Sales Data Database for based on the User Request.
    Args:
        user_query: The exact question or request from user in plain text.
    Return:
        response: Result from the lookup of the Sales Data Database.
    """
    print("Inside Lookup")

    response = query_sales_data(user_query, sales_data, df_querybot)
    
    return(response)


In [12]:
# Initialize the Inference Routing Agent
# Agent understand te Intent User Input.

def initialize_bi_agent(agent_chatbot: ChatOllama) -> object:

    agent_instruction = """
    Role: You are an extreme-precision Intent Classification Agent.

    Goals:
    1. Analyze the Semantic Meaning of the User Request and CLASSIFY it into EXACTLY ONE of the Allowed INTENT from the Intent List.
    2. If the User's Intent Matches with Any One Intent from the list, return that Value.
    3. If the User's Intent Does Not Match with value in Intent List, respond with NO_INTENT.
   
    Intent List:
    1. DBLookup: ONLY if extracting specific Numbers, Counts, List Records, or Summaries directly FROM a Sales Data.
    2. Research: Questions related to Concepts, Research, Methodology, Approach, Case Studies, Data Types, Historical Solutions, Algorithms.

    Examples:
        User Request = How was Big Data used for Walmart Marketing Analysis ?
        Return: Research

        User Request: Get me count info on Males who bought 'Widget A'.
        Return: DBLookup
    """

    agent = create_agent(
        model = agent_chatbot, 
        system_prompt=SystemMessage(content=agent_instruction)
        )
    
    return(agent)


Load Contents
- Sales Data to Pandas
- Parse PDF and Insert to RAG Database

In [13]:
# Load Sales Data to Pandas

sales_df = pd.read_csv("C:\\RanjithC\\AIProjects\\PromptEngg\\huggingface\\data\\simpli_learn\\sales_data\\biz_sales_data.csv", sep=",")
sales_df.head(5)

,Date,Product,Region,Sales,Customer_Age,Customer_Gender,Customer_Satisfaction
0,1/1/2022,Widget C,South,786,26,Male,2.874407
1,1/2/2022,Widget D,East,850,29,Male,3.365205
2,1/3/2022,Widget A,North,871,40,Female,4.547364
3,1/4/2022,Widget C,South,464,31,Male,4.555420
4,1/5/2022,Widget C,South,262,50,Female,3.982935


In [14]:
# Load Documents into Vector Store using LlamaCloud Parser
# Each file is Read, uploaed to LLamaParse Cloud.
# Files are not Cached, generate Markdowns from PDF.
# Each File is processed one at a time in a loop.

async def parse_docs(client: LlamaCloud):
    folder_path = Path("C:\\RanjithC\\AIProjects\\PromptEngg\\huggingface\\data\\simpli_learn\\rag_data\\")
    file_list = [str(file) for file in folder_path.glob("*.pdf")]

    # Read Each File in a Loop & Parse to multiple Docs
    for file_path in file_list:
        print("File Parsing: ", file_path)

        upload_file = client.files.create(file=file_path, purpose="parse")
        parse_result = client.parsing.parse(
            file_id =  upload_file.id, 
            tier = "cost_effective", 
            version = "latest",
            disable_cache = True,
            expand = ["markdown", "metadata"]
        )

        for doc in parse_result.markdown.pages:
            print("Upload: ", doc)
            insert_parsed_doc(Path(file_path).name, doc.page_number, doc.markdown)

    return(True)

await parse_docs(llamaClient)

File Parsing:  C:\RanjithC\AIProjects\PromptEngg\huggingface\data\simpli_learn\rag_data\Business_Intelligent_Approaches.pdf
Upload:  MarkdownPageMarkdownResultPage(markdown='American Journal of Scientific Research\nISSN 1450-223X Issue 50 (2012), pp. 62-75\n© EuroJournals Publishing, Inc. 2012\nhttp://www.eurojournals.com/ajsr.htm\n\n# Review Study: Business Intelligence Concepts and Approaches\n\n**Saeed Rouhani**\n*Islamic Azad University, Firoozkooh Branch*\n*Department of Industrial Engineering, Firoozkooh, Iran*\nE-mail: SRouhani@iust.ac.ir\nTel: +98-912-2034980\n\n**Sara Asgari**\n*MehrAlborz University, Tehran, Iran*\nE-mail: sara.asgary29@gmail.com\n\n**Seyed Vahid Mirhosseini**\n*MehrAlborz University, Tehran, Iran*\nE-mail: vmirhosseini@gmail.com\n\n### Abstract\n\nIn today’s challenging business environment, it is a vital for organization to access useful information and knowledge. Business Intelligence (BI) is an umbrella concept for tools, techniques and solutions that hel

True

Initialise & Chat with Agent.

In [15]:
# Format Agent Response

def format_agent_response(ai_msg):
    ai_msg = (ai_msg['messages'][1]).content.split("<|im_start|>assistant")[-1]
    return(ai_msg.strip())

In [23]:
# Initiate Chat with Agent

exit_flag = False

# Press 'Escape' Key to End the While Loop below
def on_key_press(event):
    if event.name == 'esc':
        print("Escape key pressed! Exiting input.")
        global exit_flag
        exit_flag = True
        return True  # Stop the keyboard listener

# Keyboard Listener
keyboard.on_press(on_key_press)

# Initialise all LLMs
llama_model = "llama3.1:8b-instruct-q4_K_M"
qwen_model = "qwen2.5-coder:7b-instruct-q4_K_M"
rag_agent_llm = create_rag_llm(llama_model)
bi_agent_llm = create_biagent_llm(qwen_model)
sales_query_llm = create_salesquery_llm(qwen_model)

# Initialise Agent
bi_agent = initialize_bi_agent(bi_agent_llm)

while(not exit_flag):
    user_input = input("Hi, I am Data Analyst Assistant. " \
    "I can help with 'Document Research' or Querying Sales Report. " \
    "Please enter your Question: ")

    print("User: ", user_input, flush=True)
    if (not exit_flag):
        user_prompt = {"messages": [HumanMessage(content=user_input)]}
        bi_agent_response = bi_agent.invoke(user_prompt)
        bi_agent_intent = format_agent_response(bi_agent_response)
        if (bi_agent_intent == 'Research'):
            print("Assistant - Reasearch lookup: ", search_research_data(user_input, rag_agent_llm), flush=True)
        elif (bi_agent_intent == 'DBLookup'):
            print("Assistant - SalesData lookup: ", lookup_sales_data(user_input, sales_df, sales_query_llm), flush=True)
        else:
            print("Assistant: ", "Didn't understand Request, Please let me know, you need to help with Document Research or Analysis on SalesData.", flush=True)


User:  Find details on Datasets collected for IoT Predections
Inside Search
Assistant - Reasearch lookup:  Answer:
The datasets were collected from temperature and gas sensors in the national capital region New Delhi, specifically in the month of July-September 2018. The dataset is comprised of values of AQI component and ppm concentration of different polluting gases, i.e., carbon monoxide, carbon dioxide, ammonia, and acetone. The IoT system helped to collect data with the help of sensors without human intervention.
User:  
